In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

In [2]:
train=pd.read_csv("../data/case1/train1.csv")
test=pd.read_csv("../data/case1/test1.csv")
for df in (train, test):
    df["n"] = (df["speed_rpm"] / 1000)
    df["Q"] = df["flow_m3_h"]
    df["P"] = df["power_w"]
    df["deltaP"] = (df["outlet_pressure_bar"] - df["inlet_pressure_bar"])

features = ["n", "P", "deltaP"]
X, y= train[features], train["Q"]
X_test, y_test= test[features], test["Q"]

In [3]:
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.8, random_state=42)

In [27]:
def find_split(X, y):
    features = X.columns
    Best_MSE = np.inf
    Best_feature = None
    Best_split = None
    Best_avg = None
    for feature in features:
        sort_feat = sorted(X[feature])
        for i in range(len(sort_feat)-1):
            if sort_feat[i] == sort_feat[i+1]:
                continue
            split = (sort_feat[i] + sort_feat[i+1]) / 2

            mask = X[feature] < split

            left = y[mask]
            right = y[~mask]

            avg = np.average(y)

            left_avg = np.average(left)
            right_avg = np.average(right)

            MSE_left = np.sum((left - left_avg)**2) / len(left) 
            MSE_right = np.sum((right - right_avg)**2) / len(right)
            MSE = (MSE_left*len(left)+MSE_right*len(right))/len(y)

            if MSE < Best_MSE:
                Best_MSE = MSE
                Best_feature = feature
                Best_split = split
                Best_avg = avg

    return(
        Best_MSE,
        Best_feature,
        Best_split,
        Best_avg
    )

In [43]:
def Tree(X, y, depth, min_leaf):
    MSE, feature, split, avg = find_split(X,y)

    is_leaf = 'No'
    if len(X) < min_leaf or depth == 0:
        is_leaf = 'Yes'
    if is_leaf == 'Yes':
        return {
                "leaf": True,
                "prediction": avg,
                "n_samples": len(y),
                "mse": MSE,
            }
    mask = X[feature] < split
    
    left = y[mask]
    right = y[~mask]

    X_left = X[mask]
    X_right = X[~mask]
    return {
            "leaf": False,
            "feature": feature,
            "threshold": split,
            "n_samples": len(y),
            "mse": MSE,
            "left": Tree(X_left, left, depth - 1, min_leaf)
            ,
            "right": Tree(X_right, right, depth - 1, min_leaf)
        }


In [46]:
def print_tree(node, spacing=""):
    if node["leaf"]:
        print(
            f"{spacing}Predict flow_m3_h = {node['prediction']:.2f} "
            f"(samples: {node['n_samples']}; MSE: {node['mse']:.3f})"
        )
        return

    print(
        f"{spacing}{node['feature']} < {node['threshold']:.2f}? "
        f"(samples: {node['n_samples']})"
    )

    print(f"{spacing}--> True:")
    print_tree(node["left"], spacing + "  ")

    print(f"{spacing}--> False:")
    print_tree(node["right"], spacing + "  ")


def total_leaf_squared_error(node):
    if node["leaf"]:
        return node["mse"] * node["n_samples"]

    return (
        total_leaf_squared_error(node["left"])
        + total_leaf_squared_error(node["right"])
    )
print_tree(Tree(X_train, y_train, 3, 3))

P < 1084.79? (samples: 384)
--> True:
  P < 329.79? (samples: 295)
  --> True:
    deltaP < 0.19? (samples: 134)
    --> True:
      Predict flow_m3_h = 16.66 (samples: 41; MSE: 15.444)
    --> False:
      Predict flow_m3_h = 8.07 (samples: 93; MSE: 25.481)
  --> False:
    deltaP < 0.64? (samples: 161)
    --> True:
      Predict flow_m3_h = 27.64 (samples: 74; MSE: 33.173)
    --> False:
      Predict flow_m3_h = 11.18 (samples: 87; MSE: 41.533)
--> False:
  deltaP < 0.83? (samples: 89)
  --> True:
    n < 4.37? (samples: 27)
    --> True:
      Predict flow_m3_h = 42.37 (samples: 17; MSE: 8.199)
    --> False:
      Predict flow_m3_h = 51.23 (samples: 10; MSE: 3.330)
  --> False:
    deltaP < 1.42? (samples: 62)
    --> True:
      Predict flow_m3_h = 30.66 (samples: 45; MSE: 21.348)
    --> False:
      Predict flow_m3_h = 18.11 (samples: 17; MSE: 17.152)
